**Richard Davies, Finn McEvoy, Josh Hellings** - Automated Data Visualisation for Policymaking.

# Altair: an introduction

Altair is declarative visualisation tool for creating Vega-Lite charts within Python. The Altair package provides a Python API for generating validated Vega-Lite chart specifications, i.e. JSON.

> See the Altair documentation [here](https://altair-viz.github.io/index.html)

Because of this, Altair and Vega-Lite have a (nearly) complete one to one mapping of functionality. Once you've learned how JSON chart components look in Altair Python code, it will be easy to switch between the two.

<br>

**First, import the modules we'll be using.**

In [ ]:
# 1. PREPARATORY STEPS - ACCESS PACKAGES WE NEED

## // The "altair" package, for creating Vega-Lite chart specs in Python
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites. (Locally, this just checks you're up to date.)
%pip install altair --upgrade --quiet
import altair as alt
%pip install vl-convert-python --quiet  ## Installs vl-convert-python into our Python instance, which let's Altair save charts as images (PNG, SVG, PDF). Adding '--quiet' stops it printing loads of text

## // The "pandas" package, for working with tabular data.
import pandas as pd

## // The "requests" package, for retrieving information from the web.
import requests

## // The "json" package, for helping us: make JSON easier to read, converting to JSON from Python data (dictionaries).
import json

---

## Section 1. **Building blocks**

1. Data

Key object in Altair is `Chart()`, which takes a DataFrame, URL, (or Altair Data object, which we'll see in mapping) as a single argument

In [ ]:
url = 'https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week03/data/data_shiller_pe.csv'
chart = alt.Chart(url)    # Defined a Chart object, but we haven't told it to do anything with the data

2. Mark type

In [ ]:
chart = alt.Chart(url).mark_line()    # Say how the data should be visualised. With Chart().mark_* (.mark_line() .mark_bar() .mark_point() etc)

3. Encoding our data.

Map some encoding channels to the columns in our dataset (check the URL above for what columns we have)

In [ ]:
alt.Chart(url).mark_line().encode(
    x=alt.X('Date_iso:T')     # Give the column name, and the data type ('Q' for quantitative, 'N' for nominal , 'O' for ordinal, 'T' for temporal), separated by a colon.
)

Let's separate our data by another attribute, this time we'll map it to the y-axis.

In [ ]:
alt.Chart(url).mark_line().encode(
    x=alt.X('Date_iso:T'),
    y=alt.Y('Price_real:Q')
)

So, we've seen the three building blocks for an Altair chart (just like in Vega-Lite):
- Specifying the data source in the `Chart()` object.
- Specifying the mark type
- Encoding some visual properties on columns in our dataset.


<br>
<br>

Now, let's move onto how we can display a Vega-Lite chart using Altair, and get to building some more Altair charts.

<br>

---

<br>

**Useful commands**
```python
## // Save an altair chart object
chart.save('chartName.json')                    # Save as Vega-Lite JSON spen
chart.save('chartName.png')                     # Save as PNG (resolution will be chart dimensions), make sure vl-convert-python is installed
chart.save('chartName_2x.png', scale_factor=2)  # Save as PNG with double resolution
chart.save('chartName.svg')                     # Save as SVG

## // Viewing a Vega-Lite chart spec
specAsChart = alt.Chart.from_json(specJSON)     # After loading in some JSON

## // View the chart JSON code
chart.to_json()

## // LARGE DATASETS. If your dataset has over 5000 rows, Altair will throw an error. Fix this error by running this command.
alt.data_transformers.disable_max_rows()
```

<br>

---

<br>

## Section 2. Displaying a Vega-Lite chart

##### Access a chart specification that I like

Suppose that you see a chart you like on the library page of our website. https://rdeconomist.github.io/library.

Here is a spec that we might want to use:
https://github.com/RDeconomist/RDeconomist.github.io/blob/main/charts/library/chartLine0.json.

And here is the RAW file:
https://raw.githubusercontent.com/RDeconomist/RDeconomist.github.io/main/charts/library/chartLine0.json

Lets download this data, view, and display it.


In [ ]:
# 2.  ACCESSING AND EXAMINING MY CHART SPEC:

## // Define my target URL (note that this is the RAW file)
url = "https://raw.githubusercontent.com/RDeconomist/RDeconomist.github.io/main/charts/library/chartLine0.json"

## // Get this:
specDownload = requests.get(url).json()

## // Turn it into JSON:
specJSON = json.dumps(specDownload)

# // Turn the json into an Altair chart and display it:
specAsChart = alt.Chart.from_json(specJSON)


## DISPLAY THIS ##

## // First, just the data (no formatting)
print(specDownload)
print('\n') ## This just adds a blank line so we can see what is going on
print(specJSON)     # Formatted as JSON (notice the double quotes)
print('\n') ## This just adds a blank line so we can see what is going on
specAsChart.display()       # This is like a print statement, but for Altair charts

So we've managed to pull a chart JSON spec from the web, and used Altair to display it. Now, let's learn how to make our own charts in Altair.

<br>

---

<br>

## Section 3. Charting in Altair



### Chart 1. Inline data

Let's make the same simple bar chart from saw in the Vega-Lite [examples](https://vega.github.io/vega-lite/examples/bar.html).

<br>

First, we must define some data in a `pd.DataFrame()`. Note, what we give to the function is a dictionary (i.e. JSON structure in Python), with two keys (our column headers) and an array of values for each key.

In [ ]:
# Start by creating a dataframe, defining our own data.
source = pd.DataFrame({
        'a': ['A', 'B', 'C', 'D', 'E', 'F'],
        'b': [28, 55, 43, 91, 81, 53]           # TODO: TRY ADDING A THIRD COLUMN (Tip: each column must have the same number of values)
})
source

Now we have some data, we can pass this to the `.Chart()` object. We must also specify:
- `.mark_bar()`: our mark type.
- `.encode()`: map between encoding channels (such as `x`, `y`, `color`, `size` etc) and columns in our dataset.

Notice how we chain these commands together (after the first .Chart(), the order doesn't matter). If we want to add properties to customise the 'mark', such as colour, we can add these inside `.mark_bar(color='red')`

In [ ]:
alt.Chart(source).mark_bar(color='red').encode(
    x=alt.X('a:N').axis(labelAngle=0, grid=False),
    y=alt.Y('b:Q')
)

<br>
<br>
<br>

### Chart 2. Linked data

Like we've seen in Vega-Lite, we don't need to write the data out, instead we can provide a URL to the data source. (Note: if you've got data)


Data source:
- Government, debt to GDP ratio: https://raw.githubusercontent.com/EconomicsObservatory/courses/refs/heads/main/1/s1_example_data/gov_debt_gdp_ratio.csv

In [ ]:
# Define our URL
url = 'https://raw.githubusercontent.com/EconomicsObservatory/courses/refs/heads/main/1/s1_example_data/gov_debt_gdp_ratio.csv'

# Create our chart. Using 'url' as our data source.
alt.Chart(url).mark_bar().encode(
    x=alt.X('Debt Ratio:Q'),
    y=alt.Y('Country:N')
)

> Hint: if you want to view or edit the data before building the chart, just load the data into a Pandas dataframe first using the steps we've shown previously. Then pass the DataFrame to the chart object, or save the cleaned data somewhere and link to that.

<br>
<br>

### Chart 2b. **Improvements**

Our charts working, but there's lots of visual improvements we can make.

1. Remove redunant information
2. Sort the bars according to their values
3. Format the x-axis as a percentage
4. Add a chart title
5. Change chart dimensions (height, width)
6. Adjust the colours
7. Format labels, axes, grid
8. Save


In [ ]:
# 1. Remove the axis titles
c1 = alt.Chart(url).mark_bar().encode(
    x=alt.X('Debt Ratio:Q').axis(title=''),
    y=alt.Y('Country:N').axis(title='')
)
c1.display()        # Notice, we're now saving our chart as a variable. Then we use the .display() function to view the chart. This is like a print statement, but for Altair charts

<br>

In [ ]:
# 2. Sort the bars according to their values
c2  = alt.Chart(url).mark_bar().encode(
    x=alt.X('Debt Ratio:Q').axis(title=''),
    y=alt.Y('Country:N').axis(title='').sort('-x')       # We apply the sort to the y-axis, and tell it to sort by the x-axis values. '-' means descending.
)
c2.display()

<br>

In [ ]:
# 3. Format the x-axis as a percentage
c3 = alt.Chart(url).mark_bar().encode(
    x=alt.X('Debt Ratio:Q').axis(title='', format='%'),
    y=alt.Y('Country:N').axis(title='').sort('-x')       # We apply the sort to the y-axis, and tell it to sort by the x-axis values. '-' means descending.
)
c3.display()

<br>

Chart titles are added by passing the `title` keyword argment with the data (in the `alt.Chart()` object):
- `alt.Chart(url, title=alt.Title(<TITLE ATTRIBUTE>)`

In [ ]:
# 4. Add a chart title
c4 = alt.Chart(url, title=alt.Title(
    text="Large countries: Debt ratios",
    subtitle=["Debt-to-GDP, 21 largest countries by population", "Source: World Bank"], # Subtitle is a list, so we can add multiple lines
    fontSize=16,
    anchor="start",     # Anchor title to left of chart ('end' for right)
    frame="group"       # Start from the chart's frame
)).mark_bar().encode(
    x=alt.X('Debt Ratio:Q').axis(title='', format='%'),
    y=alt.Y('Country:N').axis(title='').sort('-x')
)
c4.display()

<br>

Adjust the chart dimensions (width, height)

In [ ]:
# 5. Adjust the colours
c5 = alt.Chart(url, title=alt.Title(
    text="Large countries: Debt ratios",
    subtitle=["Debt-to-GDP, 21 largest countries by population", "Source: World Bank"],
    fontSize=16,
    anchor="start",
    frame="group"
)).mark_bar().encode(
    x=alt.X('Debt Ratio:Q').axis(title='', format='%'),
    y=alt.Y('Country:N').axis(title='').sort('-x')
).properties(
    width=360,          # Set chart width, in pixels
    height=300          # Set chart height, in pixels
)
c5.display()

<br>

Adjusting the colours of the bars and the axis labels.

In [ ]:
# 6. Adjust the colours
c6 = alt.Chart(url, title=alt.Title(
    text="Large countries: Debt ratios",
    subtitle=["Debt-to-GDP, 21 largest countries by population", "Source: World Bank"],
    fontSize=16,
    anchor="start",
    frame="group",
    subtitleColor='#596870' # Set subtitle to grey to de-emphasise
)).mark_bar(
    color='#36b7b4'     # We are not encoding the colour on any data, so we just set it directly in the mark_bar() function
).encode(
    x=alt.X('Debt Ratio:Q').axis(title='', format='%', labelColor='#596870'),
    y=alt.Y('Country:N').axis(title='', labelColor='#596870').sort('-x')
).properties(
    width=360,
    height=300
)
c6.display()

<br>

Format axis labels, grid lines, and domain

In [ ]:
# 7. Adjust the colours
c7 = alt.Chart(url, title=alt.Title(
    text="Large countries: Debt ratios",
    subtitle=["Debt-to-GDP, 21 largest countries by population", "Source: World Bank"],
    fontSize=16,
    anchor="start",
    frame="group",
    subtitleColor='#596870'
)).mark_bar(
    color='#36b7b4'
).encode(
    x=alt.X('Debt Ratio:Q').axis(
        title='', format='%', labelColor='#596870',
        labelFontSize=13, domainOpacity=0.5, tickOpacity=0.5
    ),  # Remove the black domain line
    y=alt.Y('Country:N').axis(title='', labelColor='#596870', labelFontSize=13).sort('-x')
).properties(
    width=360,
    height=300
).configure_view(
    strokeWidth=0
)
c7.display()

<br>

To save a chart, we must store the chart object as a variable, then use Altair's built-in `.save()` function.

In [ ]:
c7.save('p21_debt_ratio_bars.json')
c7.save('p21_debt_ratio_bars.png', scale_factor=2)  # save as higher res